In [ ]:
# @title Install pub-dialogue package
# Local: run 'pip install -e .' once in the repo root.
# Colab:  installs directly from GitHub.
try:
    import pub_dialogue
except ImportError:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/mlatcl/pub-dialogue.git",
    ])
    import pub_dialogue

In [ ]:
# @title Import libraries and define configuration
import os
import json
import time
import re
import random
from pathlib import Path
from collections import Counter
from datetime import datetime
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from openai import OpenAI
from IPython.display import display, HTML

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Model Configuration
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

# Clustering - now 75 clusters for concern phrases (smaller units than paragraphs)
N_CONCERN_CLUSTERS = 75
SOFT_MEMBERSHIP_THRESHOLD = 0.3

# Chunking
MIN_CHUNK_WORDS = 40
MAX_CHUNK_WORDS = 500
MIN_CHUNK_CHARS = 80

# When paragraph-level splitting (the default) fails to produce reasonable
# chunks (because the PDF lacks double-newline paragraph separators),
# fall back to sentence-level splitting and repack into chunks of about
# this many words. v18: addresses an issue identified in v17 where 20
# documents (mostly AI policy reports from 2020+) were silently truncated
# because their full text was treated as a single 500-word paragraph.
SENTENCE_FALLBACK_TARGET_WORDS = 300
SENTENCE_FALLBACK_MIN_PARAGRAPHS = 3  # if Method 1 yields fewer paragraphs, fall back

# Processing
EMBEDDING_BATCH_SIZE = 100
EXTRACTION_BATCH_SIZE = 10  # For concern extraction

# Paths
PDF_FOLDER = Path("/content/pdfs")
OUTPUT_FOLDER = Path("/content/outputs")
CHECKPOINT_FOLDER = Path("/content/checkpoints")

for folder in [PDF_FOLDER, OUTPUT_FOLDER, CHECKPOINT_FOLDER]:
    folder.mkdir(exist_ok=True)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

import warnings
warnings.filterwarnings('ignore')

print("Configuration loaded")
print(f"  Embedding model: {EMBEDDING_MODEL}")
print(f"  Target clusters: {N_CONCERN_CLUSTERS}")
print(f"  LLM for extraction: {LLM_MODEL}")

In [ ]:
# @title Load dialogue_utils — shared helper functions
# All helper functions (show_status, show_complete, show_warning,
# save_checkpoint, load_checkpoint, extract_chunks_from_pdf, extract_phrases,
# label_cluster, normalized_entropy, run_sensitivity, etc.) are imported from
# dialogue_utils.py.  The module must be on sys.path (it is fetched in the
# cell below).

try:
    import pub_dialogue.utils as du
    from pub_dialogue.utils import (
        show_status, show_complete, show_warning,
        save_checkpoint, load_checkpoint,
        CROSSCUTTING_ENTROPY_THRESHOLD,
        EXTRACTION_PROMPT, BENEFIT_EXTRACTION_PROMPT,
        ExtractionResult, extract_phrases, validate_extraction_cache,
        write_extraction_diagnostics,
        filter_missing_source_text, get_embeddings_batch,
        label_cluster, pretty_label, clusters_to_labels,
        clusters_to_lenses, html_escape,
        normalized_entropy, hhi, topk_share, parse_year, tokenize,
        is_privacy_text, entropy_by_year, ai_fingerprint_over_crosscut,
        run_sensitivity,
        vocabulary_frequency_diagnostic, generate_validation_summary,
        extract_chunks_from_pdf, reset_chunk_stats, get_chunk_stats,
        MIN_CHUNK_WORDS, MIN_CHUNK_CHARS, MAX_CHUNK_WORDS,
        SENTENCE_FALLBACK_TARGET_WORDS, SENTENCE_FALLBACK_MIN_PARAGRAPHS,
    )
    print("dialogue_utils imported successfully")
except ImportError as _e:
    print(f"dialogue_utils not found: {_e}")
    print("Fetch it with: !wget https://raw.githubusercontent.com/mlatcl/pub-dialogue/main/dialogue_utils.py")
    raise

In [ ]:
# @title Configure API access
import os as _os

api_key = None

# 1. Try Colab secrets (when running in Google Colab)
try:
    from google.colab import userdata as _userdata
    api_key = _userdata.get('OPENAI_API_KEY')
    if api_key:
        print("API key loaded from Colab secrets")
except Exception:
    pass

# 2. Try environment variable (when running locally)
if not api_key:
    api_key = _os.environ.get("OPENAI_API_KEY")
    if api_key:
        print("API key loaded from OPENAI_API_KEY environment variable")

# 3. Interactive prompt fallback
if not api_key:
    from getpass import getpass as _getpass
    api_key = _getpass("Enter OpenAI API key: ")
    print("API key entered manually")

client = OpenAI(api_key=api_key)

try:
    client.models.list()
    print("API connection verified")
except Exception as e:
    print(f"API error: {e}")

In [ ]:
# @title Summarise paragraph-level data quality

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Chunks by technology
tech_counts = chunks_df['technology_meta'].value_counts()
axes[0, 0].barh(tech_counts.index, tech_counts.values, color='steelblue')
axes[0, 0].set_xlabel('Number of Paragraphs')
axes[0, 0].set_title('Paragraphs by Technology')
axes[0, 0].invert_yaxis()

# Chunks by year
year_counts = chunks_df.groupby('year').size()
axes[0, 1].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Paragraphs')
axes[0, 1].set_title('Paragraphs by Year')
axes[0, 1].tick_params(axis='x', rotation=45)

# Word count distribution
axes[1, 0].hist(chunks_df['word_count'], bins=30, color='steelblue', edgecolor='white')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Paragraph Length Distribution')

# Chunks per document
doc_chunks = chunks_df.groupby('source_file').size().sort_values(ascending=False)
axes[1, 1].bar(range(len(doc_chunks)), doc_chunks.values, color='steelblue')
axes[1, 1].set_xlabel('Document Index')
axes[1, 1].set_ylabel('Paragraphs')
axes[1, 1].set_title('Paragraphs per Document')

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "data_quality_overview.png", dpi=150)
plt.show()

In [ ]:
# @title (v16) Chunk content-quality diagnostic
# Reports the proportion of kept chunks that look like bibliography fragments
# or survey-table rows, so the analyst can see the contamination rate at the
# chosen MIN_CHUNK_WORDS floor.

show_status("Running chunk content-quality diagnostic...")

import re

# Heuristics. Kept deliberately simple — these are diagnostics, not filters.
_BIB_PATTERN = re.compile(r"\b(et al\.?|doi:|http(s)?://|pp?\.\s*\d+|vol\.|isbn|issn)\b", re.IGNORECASE)
_TABLE_PATTERN = re.compile(r"%")  # any percent-sign occurrence
_CITATION_YEAR = re.compile(r"\([12][0-9]{3}\)")  # (1999) / (2024) etc.

def _looks_like_bibliography(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return bool(_BIB_PATTERN.search(text)) and bool(_CITATION_YEAR.search(text))

def _looks_like_table_row(text: str) -> bool:
    if not isinstance(text, str):
        return False
    pct_count = len(_TABLE_PATTERN.findall(text))
    # Heuristic: 2+ percent signs in a chunk strongly suggests a survey table row
    return pct_count >= 2

chunks_df["likely_bibliography"] = chunks_df["text"].apply(_looks_like_bibliography)
chunks_df["likely_table_row"] = chunks_df["text"].apply(_looks_like_table_row)

n_total = len(chunks_df)
n_bib = int(chunks_df["likely_bibliography"].sum())
n_table = int(chunks_df["likely_table_row"].sum())

print(f"Chunk content quality (n = {n_total}):")
print(f"  Likely bibliography fragment: {n_bib}  ({n_bib/n_total*100:.1f}%)")
print(f"  Likely survey-table row:      {n_table}  ({n_table/n_total*100:.1f}%)")
print()
print("These chunks are kept for analysis (the diagnostic does not filter).")
print("If the contamination rate is high, consider raising MIN_CHUNK_WORDS.")
print(f"Current floor: {MIN_CHUNK_WORDS} words / {MIN_CHUNK_CHARS} chars.")

# Write the flagged chunks for manual inspection
flagged = chunks_df[chunks_df["likely_bibliography"] | chunks_df["likely_table_row"]][
    ["chunk_id", "source_file", "word_count", "likely_bibliography", "likely_table_row", "text"]
]
flagged.to_csv(OUTPUT_FOLDER / "chunk_quality_flagged.csv", index=False)
show_complete(f"Wrote {len(flagged)} flagged chunks to chunk_quality_flagged.csv (for inspection)")
